# P8 tool-using agent: Kaggle runner

Thin wrapper: all logic lives in `src/toolagent`.

Settings: **GPU T4, Internet ON**, then *Save Version -> Save & Run All*. Runs are resumable (a finished
(model, agent, seed) is skipped). The smoke config runs with `--check`, so the notebook stops if the oracle fails a
task or the real model cannot load.

Download `p8_results.zip` from the Output tab: results, per-category table, bootstrap, failure taxonomy, every trace
(`traces/*.jsonl`) and the static trace viewer (`traces.html`). Everything is synthetic (no dataset licence).

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/p8-tool-agent
!pip install -q bitsandbytes accelerate && pip install -q -e . --no-deps

In [ ]:
!mkdir -p /kaggle/working/outputs
!python -c "import torch, transformers, bitsandbytes, accelerate, numpy, pandas, yaml; [print(m.__name__, m.__version__) for m in (torch, transformers, bitsandbytes, accelerate, numpy, pandas, yaml)]; print('cuda', torch.cuda.is_available(), torch.version.cuda)" | tee /kaggle/working/outputs/env.txt
!(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo no-gpu) | tee -a /kaggle/working/outputs/env.txt
!pip freeze | grep -iE '^(torch|transformers|bitsandbytes|accelerate|tokenizers|safetensors|huggingface.hub|numpy|pandas|pyyaml)==' | tee /kaggle/working/outputs/versions.txt

## Run `configs/smoke.yaml`

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_eval.py', '--config', 'configs/smoke.yaml', '--check', '--set', 'output_dir=/kaggle/working/outputs/smoke'], check=True)

In [ ]:
from pathlib import Path

for f in ['results.md', 'categories.md', 'bootstrap.md', 'failures.md']:
    p = Path('/kaggle/working/outputs/smoke') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Run `configs/eval.yaml`

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_eval.py', '--config', 'configs/eval.yaml', '--check', '--set', 'output_dir=/kaggle/working/outputs/eval'], check=True)

In [ ]:
from pathlib import Path

for f in ['results.md', 'categories.md', 'bootstrap.md', 'failures.md']:
    p = Path('/kaggle/working/outputs/eval') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Package results

In [ ]:
%cd /kaggle/working
!zip -qr p8_results.zip outputs -x '*/world/*'
!rm -rf outputs Artificial_Intelligence && ls -la /kaggle/working